In [ ]:
%cd /content
!git clone -q https://github.com/Nikhils-G/turnwave.git || true
%cd /content/turnwave
!git pull -q
!pip install -q -e . --no-deps sentencepiece datasets

import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

!python scripts/build_text_dataset.py --out data/text && \
 python -m turnwave.tokenizer data/text/corpus.txt checkpoints/tokenizer && \
 python -m turnwave.train \
    --train data/text/train.jsonl --val data/text/validation.jsonl \
    --tokenizer checkpoints/tokenizer/spm.model --out checkpoints/text_eot \
    --steps 6000 --batch-size 256 --num-workers 2 && \
 python -m turnwave.evaluate \
    --ckpt checkpoints/text_eot/best.pt \
    --tokenizer checkpoints/tokenizer/spm.model \
    --data data/text/test.jsonl --device cuda

/content
fatal: destination path 'turnwave' already exists and is not an empty directory.
/content/turnwave
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for turnwave (pyproject.toml) ... done
GPU OK: Tesla T4
train         11118 dialogues ->  169495 examples (87169 complete / 82326 truncated)
validation     1000 dialogues ->   15686 examples (8069 complete / 7617 truncated)
test           1000 dialogues ->   15086 examples (7740 complete / 7346 truncated)
tokenizer corpus: data/text/corpus.txt (87169 utterances)
I0000 00:00:1788110933.360982    8501 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: data/text/corpus.txt
  input_format: 
  model_prefix: checkpoints/tokenizer/spm
  model_type: BPE
  vocab_size: 8192
  self_test_sample_size

In [ ]:
from google.colab import files
files.download("checkpoints/text_eot/best.pt")
files.download("checkpoints/text_eot/log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, subprocess, sys

if not os.path.isdir('/content/turnwave'):
    !git clone -q https://github.com/Nikhils-G/turnwave.git /content/turnwave
%cd /content/turnwave
!git pull -q

install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
                          '--no-deps', 'sentencepiece', 'datasets', 'soundfile',
                          'onnx', 'onnxruntime', 'onnxscript'])
assert install.returncode == 0, 'install failed - read the error above'

import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
print('GPU OK:', torch.cuda.get_device_name(0))

/content/turnwave
GPU OK: Tesla T4


In [ ]:
%cd /content/turnwave
!python scripts/build_text_dataset.py --out data/text && \
 python -m turnwave.tokenizer data/text/corpus.txt checkpoints/tokenizer && \
 python -m turnwave.train --task text \
     --train data/text/train.jsonl --val data/text/validation.jsonl \
     --tokenizer checkpoints/tokenizer/spm.model --out checkpoints/text_eot \
     --steps 3500 --batch-size 256 --num-workers 2 && \
 python scripts/build_audio_dataset.py --out data/audio \
     --max-examples 60000 --max-eval-examples 6000 && \
 python -m turnwave.train --task audio --cache data/audio \
     --out checkpoints/audio_eot --steps 4000 --batch-size 128 --num-workers 2 && \
 python -m turnwave.train --task fusion --cache data/audio \
     --tokenizer checkpoints/tokenizer/spm.model \
     --text-ckpt checkpoints/text_eot/best.pt \
     --audio-ckpt checkpoints/audio_eot/best.pt \
     --out checkpoints/fusion_eot --steps 2000 --batch-size 128 --lr 1e-3 --num-workers 2 && \
 python -m turnwave.ablate --cache data/audio \
     --tokenizer checkpoints/tokenizer/spm.model \
     --text-ckpt checkpoints/text_eot/best.pt \
     --audio-ckpt checkpoints/audio_eot/best.pt \
     --fusion-ckpt checkpoints/fusion_eot/best.pt --split test --device cuda

/content/turnwave

default/train/0000.parquet: downloading bytes:   0% 0.00/3.61M [00:00<?, ?B/s]
default/train/0000.parquet: downloading bytes:  60% 2.16M/3.61M [00:01<00:01, 1.35MB/s]
default/train/0000.parquet: downloading bytes: 100% 3.60M/3.60M [00:01<00:00, 2.09MB/s,  344kB/s  ]
default/train/0000.parquet: reconstructing file: 100% 3.61M/3.61M [00:01<00:00, 2.09MB/s,  347kB/s  ]
0000.parquet: 100% 334k/334k [00:00<00:00, 73.1MB/s]
0000.parquet: 100% 331k/331k [00:00<00:00, 296MB/s]
Generating train split: 11118 examples [00:00, 224223.80 examples/s]
Generating validation split: 1000 examples [00:00, 196740.18 examples/s]
Generating test split: 1000 examples [00:00, 216726.30 examples/s]
train         11118 dialogues ->  169495 examples (87169 complete / 82326 truncated)
validation     1000 dialogues ->   15686 examples (8069 complete / 7617 truncated)
test           1000 dialogues ->   15086 examples (7740 complete / 7346 truncated)
tokenizer corpus: data/text/corpus.txt (87169 u

In [ ]:
%cd /content/turnwave
import os; assert os.path.exists('checkpoints/fusion_eot/best.pt'), 'checkpoints missing!'
!python -m turnwave.export --ckpt checkpoints/text_eot/best.pt   --out-dir checkpoints/onnx
!python -m turnwave.export --ckpt checkpoints/audio_eot/best.pt  --out-dir checkpoints/onnx
!python -m turnwave.export --ckpt checkpoints/fusion_eot/best.pt --out-dir checkpoints/onnx
!python scripts/plot_training.py checkpoints/audio_eot/log.csv  --out docs/audio_curves.png
!python scripts/plot_training.py checkpoints/fusion_eot/log.csv --out docs/fusion_curves.png
!zip -qr turnwave_phase2.zip checkpoints docs
from google.colab import files
files.download('turnwave_phase2.zip')

/content/turnwave
/content/turnwave/turnwave/export.py:84: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)
task=text
  fp32  27.84 MB    3.64 ms  max prob delta 1.79e-07
  int8   7.20 MB    3.11 ms  max prob delta 1.86e-02
  ship: in

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('turnwave_phase2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, subprocess, sys

if not os.path.isdir('/content/turnwave'):
    !git clone -q https://github.com/Nikhils-G/turnwave.git /content/turnwave
%cd /content/turnwave
!git pull -q

install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
                          '--no-deps', 'sentencepiece', 'datasets', 'soundfile',
                          'onnx', 'onnxruntime', 'onnxscript'])
assert install.returncode == 0, 'install failed - read the error above'

import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
print('GPU OK:', torch.cuda.get_device_name(0))

/content/turnwave
GPU OK: Tesla T4


In [ ]:
# 2. Text branch (~14 min). 1,500 steps: the 3,500-step run peaked at 1,250.
!python scripts/build_text_dataset.py --out data/text && \
 python -m turnwave.tokenizer data/text/corpus.txt checkpoints/tokenizer && \
 python -m turnwave.train --task text \
     --train data/text/train.jsonl --val data/text/validation.jsonl \
     --tokenizer checkpoints/tokenizer/spm.model --out checkpoints/text_eot \
     --steps 1500 --batch-size 256 --num-workers 2


default/train/0000.parquet: downloading bytes:  12% 432k/3.61M [00:01<00:11, 266kB/s]
default/train/0000.parquet: downloading bytes:  66% 2.37M/3.61M [00:01<00:00, 1.81MB/s, 41.7kB/s  ]
default/train/0000.parquet: downloading bytes: 100% 3.60M/3.60M [00:01<00:00, 1.88MB/s,  341kB/s  ]
default/train/0000.parquet: reconstructing file: 100% 3.61M/3.61M [00:01<00:00, 1.88MB/s,  344kB/s  ]
0000.parquet: 100% 334k/334k [00:00<00:00, 74.9MB/s]
0000.parquet: 100% 331k/331k [00:00<00:00, 247MB/s]
Generating train split: 11118 examples [00:00, 212334.53 examples/s]
Generating validation split: 1000 examples [00:00, 201387.81 examples/s]
Generating test split: 1000 examples [00:00, 187967.37 examples/s]
train         11118 dialogues ->  169495 examples (87169 complete / 82326 truncated)
validation     1000 dialogues ->   15686 examples (8069 complete / 7617 truncated)
test           1000 dialogues ->   15086 examples (7740 complete / 7346 truncated)
tokenizer corpus: data/text/corpus.txt (87169 

In [ ]:
# 3. Feature cache, doubled (~10 min). --cut-offset 0.2 is the default; passed
# explicitly so the manifest and this notebook agree about what was built.
!python scripts/build_audio_dataset.py --out data/audio --cut-offset 0.2 \
    --max-examples 120000 --max-eval-examples 8000

README.md: 100% 6.47k/6.47k [00:00<00:00, 15.8MB/s]
Resolving data files: 100% 404/404 [00:00<00:00, 528375.06it/s]
Resolving data files: 100% 18/18 [00:00<00:00, 153762.67it/s]
Resolving data files: 100% 18/18 [00:00<00:00, 161319.38it/s]
Resolving data files: 100% 25/25 [00:00<00:00, 190997.45it/s]
train     : 100% 120000/120000 [05:46<00:00, 346.49ex/s]
Resolving data files: 100% 404/404 [00:00<00:00, 17141.08it/s]
Resolving data files: 100% 18/18 [00:00<00:00, 90633.22it/s]
Resolving data files: 100% 18/18 [00:00<00:00, 170039.35it/s]
Resolving data files: 100% 25/25 [00:00<00:00, 113482.25it/s]
validation:  98% 7878/8000 [00:24<00:00, 323.10ex/s]
Resolving data files: 100% 404/404 [00:00<00:00, 11730.69it/s]
Resolving data files: 100% 18/18 [00:00<00:00, 160632.92it/s]
Resolving data files: 100% 18/18 [00:00<00:00, 169657.24it/s]
Resolving data files: 100% 25/25 [00:00<00:00, 145635.56it/s]
test      :  98% 7817/8000 [00:23<00:00, 327.98ex/s]
train        120,000 examples from 75,

In [ ]:
# 4. Acoustic branch, doubled steps (~70 min). Gate: val AP must beat 0.741.
!python -m turnwave.train --task audio --cache data/audio \
    --out checkpoints/audio_eot \
    --steps 8000 --batch-size 128 --lr 3e-4 --num-workers 2

device=cuda task=audio params=3.49M (trainable 3.49M) train=120000 val=7878 pos_weight=0.59
step    250  lr 2.50e-04  train 0.4584  val 0.4024  acc 0.699  f1 0.746  ap 0.868
step    500  lr 3.00e-04  train 0.3985  val 0.3938  acc 0.724  f1 0.784  ap 0.882
step    750  lr 2.98e-04  train 0.3858  val 0.3821  acc 0.649  f1 0.642  ap 0.889
step   1000  lr 2.94e-04  train 0.3753  val 0.3726  acc 0.702  f1 0.736  ap 0.892
step   1250  lr 2.89e-04  train 0.3703  val 0.3646  acc 0.737  f1 0.786  ap 0.902
step   1500  lr 2.83e-04  train 0.3653  val 0.3789  acc 0.752  f1 0.817  ap 0.904
step   1750  lr 2.75e-04  train 0.3578  val 0.3612  acc 0.691  f1 0.700  ap 0.907
step   2000  lr 2.67e-04  train 0.3505  val 0.3568  acc 0.752  f1 0.793  ap 0.907
step   2250  lr 2.57e-04  train 0.3475  val 0.3418  acc 0.742  f1 0.774  ap 0.914
step   2500  lr 2.45e-04  train 0.3461  val 0.3394  acc 0.763  f1 0.802  ap 0.915
step   2750  lr 2.33e-04  train 0.3410  val 0.3283  acc 0.758  f1 0.794  ap 0.921
step  

In [ ]:
%cd /content/turnwave
!zip -qr /content/save.zip checkpoints
from google.colab import files
files.download('/content/save.zip')

/content/turnwave


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%cd /content/turnwave
!python -m turnwave.train --task fusion --cache data/audio \
     --tokenizer checkpoints/tokenizer/spm.model \
     --text-ckpt checkpoints/text_eot/best.pt \
     --audio-ckpt checkpoints/audio_eot/best.pt \
     --out checkpoints/fusion_eot --steps 1200 --batch-size 128 --lr 1e-3 --num-workers 2 && \
 python -m turnwave.ablate --cache data/audio \
     --tokenizer checkpoints/tokenizer/spm.model \
     --text-ckpt checkpoints/text_eot/best.pt \
     --audio-ckpt checkpoints/audio_eot/best.pt \
     --fusion-ckpt checkpoints/fusion_eot/best.pt \
     --split test --device cuda --out docs/ablation.json && \
 python -m turnwave.export --ckpt checkpoints/text_eot/best.pt   --out-dir checkpoints/onnx && \
 python -m turnwave.export --ckpt checkpoints/audio_eot/best.pt  --out-dir checkpoints/onnx && \
 python -m turnwave.export --ckpt checkpoints/fusion_eot/best.pt --out-dir checkpoints/onnx
!zip -qr /content/phase4.zip checkpoints docs
from google.colab import files
files.download('/content/phase4.zip')

/content/turnwave
device=cuda task=fusion params=10.54M (trainable 0.13M) train=120000 val=7878 pos_weight=0.59
step    250  lr 8.33e-04  train 0.3174  val 0.3009  acc 0.797  f1 0.834  ap 0.937
step    500  lr 8.85e-04  train 0.2817  val 0.2961  acc 0.794  f1 0.828  ap 0.938
step    750  lr 5.07e-04  train 0.2798  val 0.2990  acc 0.805  f1 0.843  ap 0.938
step   1000  lr 1.27e-04  train 0.2806  val 0.2920  acc 0.785  f1 0.817  ap 0.939
step   1200  lr 1.00e-05  train 0.2776  val 0.2913  acc 0.797  f1 0.831  ap 0.939
done. best val AP 0.939  checkpoints in checkpoints/fusion_eot

data/audio/test: 7,817 examples (5,000 turn-final / 2,817 mid-turn)

                             acc    prec  recall      f1      ap
majority class             0.640   0.640   1.000   0.780   0.636
cue-word heuristic         0.582   0.627   0.853   0.723   0.625
text only                  0.598   0.670   0.732   0.699   0.683
audio only                 0.770   0.907   0.714   0.799   0.938
fused (text + audio)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Repo, deps, and the harness.
import os, subprocess, sys

if not os.path.isdir('/content/turnwave'):
    !git clone -q https://github.com/Nikhils-G/turnwave.git /content/turnwave
%cd /content/turnwave
!git pull -q
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
                          '--no-deps', 'sentencepiece', 'datasets', 'soundfile',
                          'onnx', 'onnxruntime', 'onnxscript'])
assert install.returncode == 0, 'install failed'
!pip install -q git+https://github.com/livekit/eot-bench
print('harness installed')

/content/turnwave
error: The following untracked working tree files would be overwritten by merge:
	docs/ablation.json
Please move or remove them before you merge.
Aborting
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 98.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.2/985.2 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 2. Unpack the trained models and confirm the adapter loads them.
if os.path.exists('/content/turnwave_phase4.zip'):
    !unzip -qo /content/turnwave_phase4.zip -d /content/turnwave
assert os.path.exists('checkpoints/onnx/fusion_eot.onnx'), \
    'no exported model - upload turnwave_phase4.zip to /content first'

os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/fusion_eot.onnx'
os.environ['TURNWAVE_TOKENIZER'] = 'checkpoints/tokenizer/spm.model'
from turnwave.eot_bench_adapter import TurnWaveAdapter
a = TurnWaveAdapter()
print('adapter OK | score_point', a.score_point, '| needs audio', a.detector.needs_audio,
      '| needs text', a.detector.needs_text)

adapter OK | score_point 0.2 | needs audio True | needs text True


In [ ]:
# 3. Score the fused model on real human-to-agent audio (~30 min, 1.9 GB).
!eot-harness predict --path livekit/eot-bench-data --name all --split validation \
    --adapter turnwave.eot_bench_adapter:TurnWaveAdapter --output-dir output

/bin/bash: line 1: eot-harness: command not found


In [ ]:

%cd /content/turnwave
!rm -f docs/ablation.json && git pull

import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'git+https://github.com/livekit/eot-bench'],
                   capture_output=True, text=True)
print("EXIT CODE:", r.returncode)
print(r.stdout[-2500:])
print("STDERR:", r.stderr[-2500:])


[Errno 2] No such file or directory: '/content/turnwave'
/content
fatal: not a git repository (or any of the parent directories): .git
EXIT CODE: 0
heel for numpy: filename=numpy-1.26.4-cp313-cp313-linux_x86_64.whl size=6917450 sha256=fcda9286a0c79062e3bacb9375ee7d09befbfee001874e0c9bb7879457092d43
  Stored in directory: /root/.cache/pip/wheels/8b/2d/9f/b6b46373f328e2ef50388915d351ccacbedac929459b5459bf
Successfully built eot-bench numpy
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
  Attempting uninstall: opentelemetry-semantic-conventions


In [ ]:
# 1. Repo, deps, and the harness.
#
# Order and flags both matter here. eot-bench pins numpy<2, so it is installed
# FIRST and everything after uses --no-deps or an explicit numpy<2, otherwise a
# later resolve pulls numpy 2.x back in and the harness breaks at import time.
import os, subprocess, sys

def run(cmd, what):
    """Fail loudly. A quiet pip failure here surfaces later as a baffling
    'command not found', which is exactly how this cell wasted an hour once."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:]); print(result.stderr[-2000:])
        raise SystemExit(f'{what} failed (exit {result.returncode})')
    print(f'{what} OK')

if not os.path.isdir('/content/turnwave'):
    run(['git', 'clone', '-q', 'https://github.com/Nikhils-G/turnwave.git',
         '/content/turnwave'], 'clone')
%cd /content/turnwave
# Generated files from an earlier run can block a pull; the tracked copies win.
subprocess.run(['git', 'checkout', '--', '.'])
subprocess.run(['rm', '-f', 'docs/ablation.json'])
subprocess.run(['git', 'pull', '-q'])

pip = [sys.executable, '-m', 'pip', 'install', '-q']
run(pip + ['git+https://github.com/livekit/eot-bench'], 'eot-bench')
run(pip + ['--no-deps', '-e', '.'], 'turnwave')
run(pip + ['sentencepiece', 'onnxruntime', 'numpy<2'], 'runtime deps')

import numpy
assert numpy.__version__.startswith('1.'), (
    f'numpy is {numpy.__version__}; eot-bench needs <2. Restart the runtime and '
    f'run this cell again.')
import shutil
assert shutil.which('eot-harness'), 'eot-harness not on PATH'
print('numpy', numpy.__version__, '| harness ready')

clone OK
/content/turnwave
eot-bench OK
turnwave OK
runtime deps OK


AssertionError: numpy is 2.1.3; eot-bench needs <2. Restart the runtime and run this cell again.

In [ ]:
%cd /content/turnwave
!mkdir -p checkpoints/onnx checkpoints/tokenizer
!unzip -qo /content/bench_models.zip -d /tmp/models
!cp /tmp/models/*.onnx checkpoints/onnx/
!cp /tmp/models/spm.model checkpoints/tokenizer/

import os
assert os.path.exists('checkpoints/onnx/fusion_eot.onnx'), 'unzip failed'
os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/fusion_eot.onnx'
os.environ['TURNWAVE_TOKENIZER'] = 'checkpoints/tokenizer/spm.model'
from turnwave.eot_bench_adapter import TurnWaveAdapter
a = TurnWaveAdapter()
print('adapter OK | score_point', a.score_point)


/content/turnwave
error [/content/bench_models.zip]:  missing 6291456 bytes in zipfile
  (attempting to process anyway)
error: invalid zip file with overlapped components (possible zip bomb)
cp: cannot stat '/tmp/models/*.onnx': No such file or directory
cp: cannot stat '/tmp/models/spm.model': No such file or directory


AssertionError: unzip failed

In [ ]:

from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/bench_models.zip" /content/
!ls -l /content/bench_models.zip


Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/bench_models.zip': No such file or directory
-rw-r--r-- 1 root root 51949618 Aug 31 13:22 /content/bench_models.zip


In [ ]:
%cd /content/turnwave
!mkdir -p checkpoints/onnx checkpoints/tokenizer
!unzip -o /content/bench_models.zip -d /tmp/models
!cp /tmp/models/*.onnx checkpoints/onnx/ && cp /tmp/models/spm.model checkpoints/tokenizer/

import os, numpy, shutil
assert os.path.exists('checkpoints/onnx/fusion_eot.onnx'), 'unzip failed'
assert numpy.__version__.startswith('1.'), f'numpy {numpy.__version__} - restart runtime, re-run setup'
assert shutil.which('eot-harness'), 'harness missing - re-run setup cell'
os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/fusion_eot.onnx'
os.environ['TURNWAVE_TOKENIZER'] = 'checkpoints/tokenizer/spm.model'
from turnwave.eot_bench_adapter import TurnWaveAdapter
print('ready | score_point', TurnWaveAdapter().score_point)


/content/turnwave
Archive:  /content/bench_models.zip
error [/content/bench_models.zip]:  missing 6291456 bytes in zipfile
  (attempting to process anyway)
error: invalid zip file with overlapped components (possible zip bomb)
cp: cannot stat '/tmp/models/*.onnx': No such file or directory


AssertionError: unzip failed

In [ ]:
%cd /content/turnwave
!rm -f /content/bench_models.zip
!mkdir -p checkpoints/onnx checkpoints/tokenizer
BASE = "https://github.com/Nikhils-G/turnwave/releases/download/models-v1"
!wget -q {BASE}/fusion_eot.onnx    -O checkpoints/onnx/fusion_eot.onnx
!wget -q {BASE}/audio_eot.onnx     -O checkpoints/onnx/audio_eot.onnx
!wget -q {BASE}/text_eot.int8.onnx -O checkpoints/onnx/text_eot.int8.onnx
!wget -q {BASE}/spm.model          -O checkpoints/tokenizer/spm.model

import os, numpy, shutil
assert os.path.getsize('checkpoints/onnx/fusion_eot.onnx') == 42353552, 'download incomplete'
assert numpy.__version__.startswith('1.'), f'numpy {numpy.__version__} - restart runtime, re-run setup'
assert shutil.which('eot-harness'), 'harness missing - re-run setup cell'
os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/fusion_eot.onnx'
os.environ['TURNWAVE_TOKENIZER'] = 'checkpoints/tokenizer/spm.model'
from turnwave.eot_bench_adapter import TurnWaveAdapter
print('ready | score_point', TurnWaveAdapter().score_point)


/content/turnwave


AssertionError: numpy 2.1.3 - restart runtime, re-run setup

In [ ]:

!pip install -q "numpy<2"
print("Done. Now: Runtime → Restart session, then run the next cell.")


Done. Now: Runtime → Restart session, then run the next cell.


In [ ]:
%cd /content/turnwave
import os, numpy, shutil
print('numpy', numpy.__version__, '| harness', shutil.which('eot-harness'))
assert numpy.__version__.startswith('1.'), 'numpy still 2.x'
assert shutil.which('eot-harness'), 'harness gone - rerun the setup cell'
os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/fusion_eot.onnx'
os.environ['TURNWAVE_TOKENIZER'] = 'checkpoints/tokenizer/spm.model'
from turnwave.eot_bench_adapter import TurnWaveAdapter
print('ready | score_point', TurnWaveAdapter().score_point)

/content/turnwave
numpy 1.26.4 | harness /usr/local/bin/eot-harness
ready | score_point 0.2


In [ ]:
!eot-harness predict --path livekit/eot-bench-data --name all --split validation \
    --adapter turnwave.eot_bench_adapter:TurnWaveAdapter --output-dir output


[eot-harness] predict: loading dataset livekit/eot-bench-data::all split=validation
README.md: 100% 3.25k/3.25k [00:00<00:00, 9.65MB/s]

data/ar/validation-00000-of-00001.parque(…): downloading bytes:  51% 74.6M/145M [00:02<00:01, 59.9MB/s, 4.74MB/s  ]
data/ar/validation-00000-of-00001.parque(…): downloading bytes:  67% 97.6M/145M [00:02<00:00, 65.3MB/s, 8.43MB/s  ]
data/ar/validation-00000-of-00001.parque(…): downloading bytes: 100% 108M/108M [00:02<00:00, 37.7MB/s, 9.69MB/s  ]
data/ar/validation-00000-of-00001.parque(…): reconstructing file: 100% 145M/145M [00:02<00:00, 50.7MB/s, 13.4MB/s  ]

data/de/validation-00000-of-00001.parque(…): downloading bytes:  48% 66.7M/138M [00:01<00:01, 61.0MB/s, 4.14MB/s  ]
data/de/validation-00000-of-00001.parque(…): downloading bytes:  65% 90.1M/138M [00:02<00:00, 75.9MB/s, 7.17MB/s  ]
data/de/validation-00000-of-00001.parque(…): downloading bytes: 100% 96.8M/96.8M [00:02<00:00, 40.1MB/s, 8.97MB/s  ]
data/de/validation-00000-of-00001.parque(…): reco

In [ ]:
import glob, os
for p in glob.glob('output/**/predictions.parquet', recursive=True):
    !eot-harness compute-metrics --predictions "{p}" --output-dir "{os.path.dirname(p)}/metrics"

run_root = 'output/livekit__eot-bench-data__validation__min_silence_100ms/en'
!eot-harness compare-models {run_root}
!cat {run_root}/comparison/report.md


# EoT Model Comparison

## Pareto Frontier

![Pareto frontier](pareto_frontier.png)

## Best Cutoff Rate at Latency Budget

![Best cutoff rate at latency budgets](cutoff_rate_at_latency_budget_300_600ms.png)

| Model | Best cutoff rate @ 0.3s latency | Best cutoff rate @ 0.6s latency |
| --- | --- | --- |
| TurnWave (from scratch) | **55.6%** | **21.7%** |
| VAD baseline | **55.6%** | **21.7%** |

## Best Latency at Cutoff Budget

![Best latency at cutoff budgets](latency_at_cutoff_budget_5_10pct.png)

| Model | Best mean latency @ 5% cutoff | Best mean latency @ 10% cutoff |
| --- | --- | --- |
| TurnWave (from scratch) | **1562 ms** | **1000 ms** |
| VAD baseline | 1600 ms | **1000 ms** |

## Operating Points

| Type | Budget | Model | Mean Latency | Cutoff | Detect | Threshold | Action Delay | Timeout |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Cutoff | 5.0% | TurnWave (from scratch) | 1.562 | 5.0% | 43.8% | 0.510 | 1.000 | 2.000 |
| Cutoff | 10.0% | TurnWave (from s

In [ ]:
import pandas as pd, glob
path = glob.glob('output/**/predictions.parquet', recursive=True)[0]
df = pd.read_parquet(path)
print(df.shape, df.columns.tolist())
print('\np_eot overall:'); print(df['p_eot'].describe())

scored = df[df['silence_dur'].round(2) == 0.2]
print(f'\nrows at the 0.2s score point: {len(scored)}')
print(scored.groupby('label')['p_eot'].agg(['count','mean','std','min','median','max']))



(11191, 7) ['id', 'language', 'span_index', 'timestamp', 'silence_dur', 'p_eot', 'label']

p_eot overall:
count    11191.000000
mean         0.680423
std          0.383923
min          0.000260
25%          0.287836
50%          0.963527
75%          0.999309
max          0.999956
Name: p_eot, dtype: float64

rows at the 0.2s score point: 1105
       count      mean       std       min    median       max
label                                                         
eot      400  0.496899  0.400401  0.000437  0.417216  0.999945
hold     705  0.385638  0.331404  0.000384  0.292083  0.999835


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
y = (scored['label'] == 'eot').astype(int)
print('AP  ', round(average_precision_score(y, scored['p_eot']), 4))
print('AUC ', round(roc_auc_score(y, scored['p_eot']), 4))
print('base', round(y.mean(), 4))


AP   0.4722
AUC  0.5629
base 0.362
